# Validación Visual Completa - Todas las Estaciones

Este notebook permite revisar visualmente cada una de las 18 estaciones para identificar problemas de imputación, outliers y anomalías.

**Objetivo:** Clasificar cada estación como:
- ✅ **EXCELLENT**: Sin problemas visibles
- ⚠️ **USABLE_WITH_CAUTION**: Problemas menores, usar con precauciones
- 🔴 **EXCLUDE**: Problemas graves, no usar para análisis de frecuencia

---

## Instrucciones:
1. Ejecutar celda por celda
2. Para cada estación, revisar las 5 figuras generadas
3. Anotar observaciones en la tabla de conclusiones
4. Al final, exportar el resumen a Markdown

In [ ]:
import sys
import os
sys.path.append('./funciones')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from graficador import generar_cartilla_completa

# Cargar datos
df = pd.read_csv('./datos/processed/Datos_Imputados_v3.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Datos cargados: {len(df):,} registros")
print(f"Estaciones: {df['station'].nunique()}")

In [ ]:
# Lista de TODAS las estaciones (18)

ESTACIONES = {
    'Zona Baja': [
        'CALAMAR',      # Historial largo, referencia
        'BARBOSA',      # Relación con Magangué
        'TENERIFE',     # ⚠️ Revisar 2010-2012
        'PLATO',        # ⚠️ Revisar inicio
        'MAGANGUE',     # Donante confiable
        'SAN_PEDRITO',  # 🔴 Terminal, sin donante
        'COYONGAL'      # 🔴 Saltos estructurales
    ],
    'Zona Media': [
        'LAS_AGUADAS',  # ⚠️ Revisar inicio
        'EL_BANCO',     # Serie estable
        'PENONCITO',    # ⚠️ Revisar inicio
        'RIO_NUEVO',    # 🔴 Extrapolación fuera de rango
        'LAS_PALOMAS',  # 🔴 Dependencia crítica de donante
        'ARMENIA'       # 🔴 Sin donante confiable
    ],
    'Zona Alta': [
        'LA_GLORIA',    # ⚠️ Revisar 2010-2012
        'EL_CONTENTO',  # ⚠️ Revisar inicio
        'SITIO_NUEVO',  # ⚠️ Revisar 2010-2012 + remanso
        'PUERTO_WILCHES', # ⚠️ Revisar 2010-2012 + serie corta
        'GAMARRA_AUT'   # 🔴 Serie corta + automática
    ]
}

# Flatten list
TODAS_ESTACIONES = [est for zona in ESTACIONES.values() for est in zona]

print(f"Total estaciones a validar: {len(TODAS_ESTACIONES)}")
for zona, ests in ESTACIONES.items():
    print(f"\n{zona}: {', '.join(ests)}")

---

## Función de Validación Visual

In [ ]:
def validar_estacion_visual(nombre_estacion, generar_figuras=True):
    """
    Valida visualmente una estación y genera figuras si se solicita.
    Retorna diccionario con observaciones.
    """
    
    print(f"\n{'='*70}")
    print(f"VALIDANDO: {nombre_estacion}")
    print(f"{'='*70}")
    
    # Cargar datos de la estación
    df_est = df[df['station'] == nombre_estacion].copy()
    
    if len(df_est) == 0:
        print(f"✗ No hay datos para {nombre_estacion}")
        return {
            'estacion': nombre_estacion,
            'estado': 'SIN_DATOS',
            'observaciones': ['No hay datos disponibles']
        }
    
    # Estadísticas básicas
    n_total = len(df_est)
    n_validos = df_est['level'].notna().sum()
    cobertura = (n_validos / n_total) * 100
    year_start = df_est['year'].min()
    year_end = df_est['year'].max()
    
    print(f"Registros: {n_validos:,} / {n_total:,} ({cobertura:.1f}%)")
    print(f"Periodo: {year_start:.0f} - {year_end:.0f}")
    
    # Generar figuras
    if generar_figuras:
        output_dir = f"./resultados/{nombre_estacion}"
        os.makedirs(output_dir, exist_ok=True)
        
        try:
            generar_cartilla_completa(nombre_estacion, output_dir=output_dir + '/')
            print(f"✓ Figuras generadas en: {output_dir}")
        except Exception as e:
            print(f"✗ Error generando figuras: {e}")
    
    # Template para observaciones (usuario debe llenar)
    observaciones = {
        'estacion': nombre_estacion,
        'n_registros': n_validos,
        'cobertura_pct': cobertura,
        'periodo': f"{year_start:.0f}-{year_end:.0f}",
        'estado': 'PENDIENTE_REVISION',  # EXCELLENT, USABLE_WITH_CAUTION, EXCLUDE
        'observaciones': [],
        'problemas_detectados': {
            'serie_tiempo': '',  # Describir problemas en Fig1
            'curva_duracion': '',  # Describir problemas en Fig3
            'qq_plot': '',  # Describir problemas en Fig5
        },
        'recomendacion': ''
    }
    
    return observaciones

print("✓ Función de validación cargada")

---

## Validación por Zonas

### ZONA BAJA (7 estaciones)

In [ ]:
# Validar Zona Baja
resultados_zona_baja = []

for estacion in ESTACIONES['Zona Baja']:
    resultado = validar_estacion_visual(estacion, generar_figuras=True)
    resultados_zona_baja.append(resultado)

### ZONA MEDIA (6 estaciones)

In [ ]:
# Validar Zona Media
resultados_zona_media = []

for estacion in ESTACIONES['Zona Media']:
    resultado = validar_estacion_visual(estacion, generar_figuras=True)
    resultados_zona_media.append(resultado)

### ZONA ALTA (5 estaciones)

In [ ]:
# Validar Zona Alta
resultados_zona_alta = []

for estacion in ESTACIONES['Zona Alta']:
    resultado = validar_estacion_visual(estacion, generar_figuras=True)
    resultados_zona_alta.append(resultado)

---

## Consolidar Resultados

In [ ]:
# Consolidar todos los resultados
todos_resultados = (
    resultados_zona_baja + 
    resultados_zona_media + 
    resultados_zona_alta
)

# Crear DataFrame
df_resultados = pd.DataFrame(todos_resultados)

print(f"Total estaciones procesadas: {len(df_resultados)}")
print("\nResumen por estado:")
print(df_resultados['estado'].value_counts())

# Mostrar tabla
display(df_resultados[['estacion', 'n_registros', 'cobertura_pct', 'estado']])

---

## Exportar a Markdown

In [ ]:
# Generar reporte en Markdown

def generar_markdown(resultados):
    """Genera reporte Markdown con conclusiones."""
    
    md = """# Reporte de Validación Visual - Estaciones MagdalenaBot

**Fecha:** {fecha}  
**Responsable:** Alexander Alberto Montalvo Molina  
**Metodología:** Revisión visual de cartillas (5 figuras por estación)

## Resumen Ejecutivo

| Categoría | N° Estaciones | Lista |
|-----------|---------------|-------|
""".format(fecha=pd.Timestamp.now().strftime('%Y-%m-%d'))
    
    # Clasificar
    excellent = [r for r in resultados if r['estado'] == 'EXCELLENT']
    usable = [r for r in resultados if r['estado'] == 'USABLE_WITH_CAUTION']
    exclude = [r for r in resultados if r['estado'] == 'EXCLUDE']
    pending = [r for r in resultados if r['estado'] == 'PENDIENTE_REVISION']
    
    md += f"| ✅ EXCELLENT | {len(excellent)} | {', '.join([r['estacion'] for r in excellent]) or 'Ninguna'} |\n"
    md += f"| ⚠️ USABLE_WITH_CAUTION | {len(usable)} | {', '.join([r['estacion'] for r in usable]) or 'Ninguna'} |\n"
    md += f"| 🔴 EXCLUDE | {len(exclude)} | {', '.join([r['estacion'] for r in exclude]) or 'Ninguna'} |\n"
    md += f"| ⏳ PENDIENTE | {len(pending)} | {', '.join([r['estacion'] for r in pending]) or 'Ninguna'} |\n"
    
    md += "\n## Detalle por Estación\n\n"
    
    for r in resultados:
        md += f"### {r['estacion']}\n\n"
        md += f"- **Estado:** {r['estado']}\n"
        md += f"- **Cobertura:** {r['cobertura_pct']:.1f}%\n"
        md += f"- **Periodo:** {r['periodo']}\n"
        md += f"- **Registros:** {r['n_registros']:,}\n\n"
        
        if r['observaciones']:
            md += "**Observaciones:**\n"
            for obs in r['observaciones']:
                md += f"- {obs}\n"
            md += "\n"
        
        if r['recomendacion']:
            md += f"**Recomendación:** {r['recomendacion']}\n\n"
    
    return md

# Generar Markdown
markdown_content = generar_markdown(todos_resultados)

# Guardar
output_file = './resultados/VALIDACION_VISUAL_COMPLETA.md'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(markdown_content)

print(f"✓ Reporte guardado en: {output_file}")